In [3]:
#Load all cleaned datasets into SQLite — use SQLAlchemy create_engine + df.to_sql(). Verify row counts match source CSVs.
#Write 10 analytical SQL queries — top 5 funds by AUM, average NAV per month, SIP YoY growth, transactions by state, funds with expense_ratio < 1%, and 5 more of your choice.


In [4]:
# import the libraries

In [5]:
# Load all CSV files

In [6]:
import pandas as pd

fund_master = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/01_fund_master.csv")
nav_history = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/02_nav_history.csv")
aum = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/03_aum_by_fund_house.csv")
sip = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/04_monthly_sip_inflows.csv")
category_inflows = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/05_category_inflows.csv")
folio = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/06_industry_folio_count.csv")
performance = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/07_scheme_performance.csv")
transactions = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/08_investor_transactions.csv")
holdings = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/09_portfolio_holdings.csv")
benchmark = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/10_benchmark_indices.csv")

In [7]:
#Step 2: Verify row counts before loading

In [8]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category_inflows": category_inflows,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print(name, len(df))

fund_master 40
nav_history 46000
aum 90
sip 48
category_inflows 144
folio 21
performance 40
transactions 32778
holdings 322
benchmark 8050


# Step 3: Create SQLite database

In [9]:
pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
#Import:

In [10]:
from sqlalchemy import create_engine

In [12]:
#Create database:

In [13]:
engine = create_engine("sqlite:///mutual_fund.db")

In [14]:
engine

Engine(sqlite:///mutual_fund.db)

In [19]:
import pandas as pd

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", engine)

,name
0,dim_fund
1,sqlite_sequence
2,dim_date
3,fact_nav
4,fact_transactions
5,fact_performance
6,fact_aum
7,fund_master
8,nav_history
9,aum_by_fund_house


In [20]:
print(datasets.keys())

dict_keys(['fund_master', 'nav_history', 'aum', 'sip', 'category_inflows', 'folio', 'performance', 'transactions', 'holdings', 'benchmark'])


In [15]:
# Step 4: Load DataFrames into SQLite

In [23]:
fund_master.to_sql(
    "fund_master",
    engine,
    if_exists="replace",
    index=False
)

nav_history.to_sql(
    "nav_history",
    engine,
    if_exists="replace",
    index=False
)

aum.to_sql(
    "aum",
    engine,
    if_exists="replace",
    index=False
)

sip.to_sql(
    "sip",
    engine,
    if_exists="replace",
    index=False
)

category_inflows.to_sql(
    "category_inflows",
    engine,
    if_exists="replace",
    index=False
)

folio.to_sql(
    "folio",
    engine,
    if_exists="replace",
    index=False
)

performance.to_sql(
    "performance",
    engine,
    if_exists="replace",
    index=False
)

transactions.to_sql(
    "transactions",
    engine,
    if_exists="replace",
    index=False
)

holdings.to_sql(
    "holdings",
    engine,
    if_exists="replace",
    index=False
)

benchmark.to_sql(
    "benchmark",
    engine,
    if_exists="replace",
    index=False
)

8050

In [17]:
# Step 5: Verify row counts after loading

In [24]:
from sqlalchemy import text

for table_name, df in datasets.items():

    query = f"""
    SELECT COUNT(*)
    FROM {table_name}
    """

    db_count = pd.read_sql(query, engine).iloc[0,0]

    print(
        table_name,
        "CSV:", len(df),
        "DB:", db_count
    )

fund_master CSV: 40 DB: 40
nav_history CSV: 46000 DB: 46000
aum CSV: 90 DB: 90
sip CSV: 48 DB: 48
category_inflows CSV: 144 DB: 144
folio CSV: 21 DB: 21
performance CSV: 40 DB: 40
transactions CSV: 32778 DB: 32778
holdings CSV: 322 DB: 322
benchmark CSV: 8050 DB: 8050


In [25]:
#Step 6: Explore table columns

In [26]:
print(fund_master.columns)
print(nav_history.columns)
print(transactions.columns)

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='object')
Index(['amfi_code', 'date', 'nav'], dtype='object')
Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')


In [27]:
print(aum.columns)
print(sip.columns)
print(category_inflows.columns)

Index(['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes'], dtype='object')
Index(['month', 'sip_inflow_crore', 'active_sip_accounts_crore',
       'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct'],
      dtype='object')
Index(['month', 'category', 'net_inflow_crore'], dtype='object')


In [28]:
print(folio.columns)
print(performance.columns)
print(holdings.columns)
print(benchmark.columns)

Index(['month', 'total_folios_crore', 'equity_folios_crore',
       'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore'],
      dtype='object')
Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')
Index(['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct',
       'market_value_cr', 'current_price_inr', 'portfolio_date'],
      dtype='object')
Index(['date', 'index_name', 'close_value'], dtype='object')


# Step 7: Analytical SQL Queries

In [30]:
# 1. Top 5 funds by AUM

In [47]:
    query = """
SELECT
    fund_house,
    MAX(aum_crore) AS total_aum
FROM aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;
"""

Top5fund = pd.read_sql_query(query, engine)

print(result)

      month     avg_nav
0   2022-01  207.061394
1   2022-02  207.717759
2   2022-03  209.692614
3   2022-04  211.833458
4   2022-05  212.731451
5   2022-06  213.860867
6   2022-07  213.956111
7   2022-08  215.683994
8   2022-09  218.494307
9   2022-10  219.529633
10  2022-11  223.470692
11  2022-12  226.760632
12  2023-01  230.671234
13  2023-02  233.847674
14  2023-03  238.009623
15  2023-04  240.613317
16  2023-05  241.892889
17  2023-06  244.605453
18  2023-07  245.780681
19  2023-08  247.734606
20  2023-09  251.833677
21  2023-10  255.607027
22  2023-11  258.352467
23  2023-12  262.020325
24  2024-01  264.620575
25  2024-02  266.331713
26  2024-03  269.597812
27  2024-04  271.300039
28  2024-05  273.450521
29  2024-06  275.465499
30  2024-07  279.024099
31  2024-08  279.736082
32  2024-09  281.016961
33  2024-10  282.127765
34  2024-11  285.243625
35  2024-12  285.272562
36  2025-01  288.038374
37  2025-02  291.744584
38  2025-03  296.158074
39  2025-04  299.412965
40  2025-05  303

In [48]:
Top5fund

,fund_house,total_aum
0,SBI Mutual Fund,1250000
1,ICICI Prudential MF,1074000
2,HDFC Mutual Fund,930000
3,Nippon India MF,700000
4,Kotak Mahindra MF,580000


In [40]:
# 2. Average NAV per month

In [45]:
query = """
SELECT
    strftime('%Y-%m', date) AS month,
    AVG(nav) AS avg_nav
FROM nav_history
GROUP BY month
ORDER BY month;
"""

Average_NAV = pd.read_sql_query(query, engine)

print(result.head())

     month     avg_nav
0  2022-01  207.061394
1  2022-02  207.717759
2  2022-03  209.692614
3  2022-04  211.833458
4  2022-05  212.731451


In [46]:
Average_NAV

,month,avg_nav
0,2022-01,207.061394
1,2022-02,207.717759
2,2022-03,209.692614
3,2022-04,211.833458
4,2022-05,212.731451
5,2022-06,213.860867
6,2022-07,213.956111
7,2022-08,215.683994
8,2022-09,218.494307
9,2022-10,219.529633


In [49]:
# 3. SIP YoY Growth

In [50]:
query = """
SELECT
    substr(month, 1, 4) AS year,
    SUM(sip_inflow_crore) AS total_sip
FROM monthly_sip_inflows
GROUP BY year
ORDER BY year;
"""

SIP_YOY = pd.read_sql_query(query, engine)

print(SIP_YOY)

   year  total_sip
0  2022     149437
1  2023     184763
2  2024     269781
3  2025     335740


In [51]:
SIP_YOY

,year,total_sip
0,2022,149437
1,2023,184763
2,2024,269781
3,2025,335740


In [52]:
#4. Transactions by state

In [53]:
query = """
SELECT
    state,
    COUNT(*) AS txn_count
FROM investor_transactions
GROUP BY state
ORDER BY txn_count DESC;
"""

transactions_by_state = pd.read_sql_query(query, engine)

print(transactions_by_state)

             state  txn_count
0           Punjab       2965
1   Madhya Pradesh       2931
2       Tamil Nadu       2806
3          Gujarat       2780
4      West Bengal       2748
5          Haryana       2736
6        Telangana       2718
7    Uttar Pradesh       2695
8            Delhi       2677
9        Karnataka       2621
10       Rajasthan       2577
11     Maharashtra       2524


In [54]:
# 5. Funds with expense ratio below 1%

In [55]:
query = """
SELECT
    scheme_name,
    expense_ratio_pct
FROM scheme_performance
WHERE expense_ratio_pct < 1;
"""

funds_low_expense = pd.read_sql_query(query, engine)

print(funds_low_expense)

                                          scheme_name  expense_ratio_pct
0            SBI Bluechip Fund - Direct Plan - Growth               0.66
1           SBI Small Cap Fund - Direct Plan - Growth               0.72
2        SBI Magnum Gilt Fund - Regular Plan - Growth               0.77
3            HDFC Top 100 Fund - Direct Plan - Growth               0.92
4   HDFC Mid-Cap Opportunities Fund - Direct - Growth               0.78
5        HDFC Short Term Debt Fund - Regular - Growth               0.56
6           ICICI Pru Bluechip Fund - Direct - Growth               0.80
7            ICICI Pru Liquid Fund - Regular - Growth               0.74
8       Nippon India Large Cap Fund - Direct - Growth               0.72
9                      Nippon India ETF Nifty 50 BeES               0.89
10  Nippon India Gilt Securities Fund - Regular - ...               0.55
11               Kotak Liquid Fund - Regular - Growth               0.60
12               Axis Bluechip Fund - Direct - Grow

In [56]:
# 6. Top fund houses by AUM

In [57]:
query = """
SELECT
    fund_house,
    SUM(aum_crore) AS total_aum
FROM aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;
"""

top5_aum = pd.read_sql_query(query, engine)

print(top5_aum)

            fund_house  total_aum
0      SBI Mutual Fund    8491000
1  ICICI Prudential MF    6293000
2     HDFC Mutual Fund    5732000
3      Nippon India MF    3909000
4    Kotak Mahindra MF    3502000


In [58]:
#7. Highest Sharpe Ratio

In [59]:
query = text("""
SELECT
    scheme_name,
    sharpe_ratio
FROM scheme_performance
ORDER BY sharpe_ratio DESC
LIMIT 10
""")

High_Sharpe_ratio = pd.read_sql_query(query, engine)

print(High_Sharpe_ratio)

                                         scheme_name  sharpe_ratio
0           ICICI Pru Liquid Fund - Regular - Growth          7.68
1               Kotak Liquid Fund - Regular - Growth          6.18
2                ABSL Liquid Fund - Regular - Growth          5.14
3       HDFC Short Term Debt Fund - Regular - Growth          1.84
4       SBI Magnum Gilt Fund - Regular Plan - Growth          1.52
5  Nippon India Gilt Securities Fund - Regular - ...          1.33
6          HDFC Top 100 Fund - Regular Plan - Growth          1.06
7      Mirae Asset Large Cap Fund - Regular - Growth          1.06
8          ICICI Pru Bluechip Fund - Direct - Growth          1.03
9     Nippon India Large Cap Fund - Regular - Growth          1.00


In [60]:
#Average Return by Category

In [61]:
query = text("""
SELECT
    category,
    AVG(return_3yr_pct) AS avg_return
FROM scheme_performance
GROUP BY category
ORDER BY avg_return DESC
""")

Avg_return = pd.read_sql_query(query, engine)

print(Avg_return)

           category  avg_return
0         Small Cap   21.686667
1           Mid Cap   16.590000
2         Flexi Cap   15.495000
3             Value   14.760000
4   Large & Mid Cap   14.560000
5              ELSS   13.580000
6         Large Cap   12.985714
7             Index   12.100000
8         Index/ETF   11.770000
9    Short Duration    7.370000
10           Liquid    6.333333
11             Gilt    5.690000


In [62]:
# 9. Largest Portfolio Holdings

In [64]:
query = text("""
SELECT
    stock_name,
    weight_pct
FROM portfolio_holdings
ORDER BY weight_pct DESC
LIMIT 10
""")

large_port_hold = pd.read_sql_query(query, engine)

print(large_port_hold)

                      stock_name  weight_pct
0                    Infosys Ltd       38.18
1  Sun Pharmaceutical Industries       35.07
2              Bharti Airtel Ltd       30.44
3                       NTPC Ltd       28.95
4            State Bank of India       28.61
5          Grasim Industries Ltd       28.25
6        Mahindra & Mahindra Ltd       26.07
7                      Wipro Ltd       25.90
8              Bharti Airtel Ltd       25.53
9  Sun Pharmaceutical Industries       23.46


In [65]:
# 10. Benchmark Performance Comparison

In [66]:
query = text("""
SELECT
    scheme_name,
    return_3yr_pct,
    benchmark_3yr_pct,
    (return_3yr_pct - benchmark_3yr_pct) AS excess_return
FROM scheme_performance
ORDER BY excess_return DESC
""")

excess_return_df = pd.read_sql_query(query, engine)

print(excess_return_df)

                                          scheme_name  return_3yr_pct  \
0        HDFC Short Term Debt Fund - Regular - Growth            7.37   
1       Kotak Emerging Equity Fund - Regular - Growth           18.23   
2            ICICI Pru Liquid Fund - Regular - Growth            7.68   
3              Kotak Flexicap Fund - Regular - Growth           15.65   
4              ABSL Small Cap Fund - Regular - Growth           22.38   
5          DSP Top 100 Equity Fund - Regular - Growth           12.82   
6                      Nippon India ETF Nifty 50 BeES           11.77   
7               UTI Flexi Cap Fund - Regular - Growth           15.34   
8            SBI Bluechip Fund - Direct Plan - Growth           11.30   
9   Mirae Asset Emerging Bluechip Fund - Regular -...           14.56   
10      Nippon India Large Cap Fund - Direct - Growth           12.33   
11      Mirae Asset Large Cap Fund - Regular - Growth           14.81   
12       SBI Magnum Gilt Fund - Regular Plan - Grow